In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))


**<font size="6" color="red">ch4_RNN(Recurrent Neural Network;순환신경망)</font>**
- 순서나 시간 데이터가 중요할 때 ex. 번역, 음성인식, 주가예측
# 1. 문맥을 이용하여 모델 만들기 

In [3]:
text = """ 경마장에 있는 말이 뛰고 있다. 
그의 말이 법이다. 
가는 말이 고와야 오는 말이 곱다 """
# text1 = "겨울이 오는 날 "

In [5]:
from keras_preprocessing.text import Tokenizer
t = Tokenizer()
t.fit_on_texts([text, 
                text1])
encoded = t.texts_to_sequences([text, text1])
print(encoded)
print(t.word_index)

[[3, 4, 1, 5, 6, 7, 1, 8, 9, 1, 10, 2, 1, 11], [12, 2, 13]]
{'말이': 1, '오는': 2, '경마장에': 3, '있는': 4, '뛰고': 5, '있다': 6, '그의': 7, '법이다': 8, '가는': 9, '고와야': 10, '곱다': 11, '겨울이': 12, '날': 13}


In [ ]:
text = """ 경마장에 있는 말이 뛰고 있다. 
그의 말이 법이다. 
가는 말이 고와야 오는 말이 곱다 """

In [6]:
# 문자를 인덱스 시퀀스로 변환하기 위한 과정 
from keras_preprocessing.text import Tokenizer
t = Tokenizer()
t.fit_on_texts([text]) # list 안으로 넣어야 한다. 
print(t.word_index)

{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '곱다': 11}


In [12]:
#문자열 리스트를 인덱스 시퀀스로 변환 
print(t.texts_to_sequences(['경마장에 말이 있다', '말이 뛴다']))
print(t.texts_to_sequences(['가는 말이 곱다'])[0])

[[2, 1, 5], [1]]
[8, 1, 11]


In [13]:
for key, value in t.word_index.items():
    print(key, value)

말이 1
경마장에 2
있는 3
뛰고 4
있다 5
그의 6
법이다 7
가는 8
고와야 9
오는 10
곱다 11


In [ ]:
text = """ 경마장에 있는 말이 뛰고 있다. 
그의 말이 법이다. 
가는 말이 고와야 오는 말이 곱다 """

In [20]:
# text를 학습시키기 위한 ['경마장에 있는', '경마장에 있는 말이', ....]
sequences = []
for line in text.split('\n'):
    print('원 문장:',line)
    encoded = t.texts_to_sequences([line])[0]
    print('encoded된 문장 :', encoded)
    for i in range(0, len(encoded)-1): #시작 index
        for j in range(i+2, len(encoded)+1): #끝나는 index 
            sequences.append(encoded[i:j])
#sequences
print('sequences와 해석 출력')
for sequence in sequences:
    #print(sequence)
    for word_seq in sequence:
        for key, value in t.word_index.items():
            if word_seq==value:
                print("{}:{}".format(word_seq, key), end=' ')
                break
    print()

원 문장:  경마장에 있는 말이 뛰고 있다. 
encoded된 문장 : [2, 3, 1, 4, 5]
원 문장: 그의 말이 법이다. 
encoded된 문장 : [6, 1, 7]
원 문장: 가는 말이 고와야 오는 말이 곱다 
encoded된 문장 : [8, 1, 9, 10, 1, 11]
sequences와 해석 출력
2:경마장에 3:있는 
2:경마장에 3:있는 1:말이 
2:경마장에 3:있는 1:말이 4:뛰고 
2:경마장에 3:있는 1:말이 4:뛰고 5:있다 
3:있는 1:말이 
3:있는 1:말이 4:뛰고 
3:있는 1:말이 4:뛰고 5:있다 
1:말이 4:뛰고 
1:말이 4:뛰고 5:있다 
4:뛰고 5:있다 
6:그의 1:말이 
6:그의 1:말이 7:법이다 
1:말이 7:법이다 
8:가는 1:말이 
8:가는 1:말이 9:고와야 
8:가는 1:말이 9:고와야 10:오는 
8:가는 1:말이 9:고와야 10:오는 1:말이 
8:가는 1:말이 9:고와야 10:오는 1:말이 11:곱다 
1:말이 9:고와야 
1:말이 9:고와야 10:오는 
1:말이 9:고와야 10:오는 1:말이 
1:말이 9:고와야 10:오는 1:말이 11:곱다 
9:고와야 10:오는 
9:고와야 10:오는 1:말이 
9:고와야 10:오는 1:말이 11:곱다 
10:오는 1:말이 
10:오는 1:말이 11:곱다 
1:말이 11:곱다 


In [24]:
# sequences 의 길이를 모두 같게 (padding)
my_len= max([len(seq) for seq in sequences])
my_len

6

In [30]:
# sequences를 훈련 가능하도록 6개로 만들어 주는 것(앞에 0, 뒤에 0)
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_sequences = pad_sequences(sequences, 
                                maxlen=my_len, 
                                padding='pre', # 앞에 0을 padding
                                #truncating=='post',
                                )
padded_sequences[:3], padded_sequences.shape

(array([[0, 0, 0, 0, 2, 3],
        [0, 0, 0, 2, 3, 1],
        [0, 0, 2, 3, 1, 4]]),
 (28, 6))

In [33]:
# 독립변수(X)와 타겟변수 (y)를 분리 
X= padded_sequences[:, :-1]
y= padded_sequences[:, -1]
# 단어갯수 
vocab_size = len(t.word_index)
vocab_size

11

In [35]:
# 종속변수의 원핫인토딩 
from tensorflow.keras.utils import to_categorical
Y=to_categorical(y, vocab_size+1) # argmax를 하려고 +1 해줌 
Y[:3]

array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [ ]:
# 모델 생성 ( Embedding -> RNN -> Dense층 )
# Embedding 층의 입력은 12(희소행렬) -> 출력 10. 이 단계에서 필요한 행렬